In [1]:
%load_ext autoreload

In [2]:
%autoreload 2

## Feature Engineering & Preprocessing (One-Hot + Separate Scaling)

In [3]:
import pandas as pd
import numpy as np
import sys
import os
import joblib

# Add the scripts folder to path
sys.path.append(os.path.abspath("../scripts"))

# Import our feature engineering functions
from feature_engineering import *
from eda_functions import *
from gan_util import build_feature_info

## 1. Load raw data

In [4]:
df = load_data("../data/processed/default_credit_EDA_cleaned.csv", clean_cols=True, drop_unnamed=True)

# Preview
print("Raw data shape:", df.shape)
df.head()

Raw data shape: (30000, 24)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,Class
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


## 2. Target mapping & missing values

In [5]:
df['Class'].unique()
df['Class'].value_counts()


Class
0    23364
1     6636
Name: count, dtype: int64

In [6]:

df = handle_missing_values(df)

No missing values detected. Skipping imputation.


## 3. Identify numerical / categorical columns

In [7]:
num_cols, cat_cols = get_feature_types(df, target='Class')
print("Numerical:", num_cols)
print("Categorical:", cat_cols)

Numerical: ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
Categorical: []


## 4. Auto‑detect low‑cardinality numerical columns as categorical

In [8]:
## Automatically detect low‑cardinality numerical columns as categorical
num_cols, cat_cols = auto_detect_categorical(df, num_cols, cat_cols, unique_threshold=10)
print("\nAfter auto‑detection:")
print("Numerical (continuous):", num_cols)
print("Categorical (including detected):", cat_cols)

Moving 'SEX' to categorical (unique values: 2 ≤ 10)
Moving 'EDUCATION' to categorical (unique values: 7 ≤ 10)
Moving 'MARRIAGE' to categorical (unique values: 4 ≤ 10)
Moving 'PAY_5' to categorical (unique values: 10 ≤ 10)
Moving 'PAY_6' to categorical (unique values: 10 ≤ 10)

After auto‑detection:
Numerical (continuous): ['LIMIT_BAL', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
Categorical (including detected): ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_5', 'PAY_6']


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   LIMIT_BAL  30000 non-null  int64   
 1   SEX        30000 non-null  category
 2   EDUCATION  30000 non-null  category
 3   MARRIAGE   30000 non-null  category
 4   AGE        30000 non-null  int64   
 5   PAY_0      30000 non-null  int64   
 6   PAY_2      30000 non-null  int64   
 7   PAY_3      30000 non-null  int64   
 8   PAY_4      30000 non-null  int64   
 9   PAY_5      30000 non-null  category
 10  PAY_6      30000 non-null  category
 11  BILL_AMT1  30000 non-null  int64   
 12  BILL_AMT2  30000 non-null  int64   
 13  BILL_AMT3  30000 non-null  int64   
 14  BILL_AMT4  30000 non-null  int64   
 15  BILL_AMT5  30000 non-null  int64   
 16  BILL_AMT6  30000 non-null  int64   
 17  PAY_AMT1   30000 non-null  int64   
 18  PAY_AMT2   30000 non-null  int64   
 19  PAY_AMT3   30000 non-null

## 5. One-hot encode categorical variables

In [10]:
df_encoded, cat_cols = encode_categorical(df)   # works because columns are already 'category'
print("Shape after one‑hot:", df_encoded.shape)


# Verify that InstallmentRate was one‑hot expanded
print([c for c in df_encoded.columns if 'InstallmentRate' in c])
df_encoded.head()


Shape after one‑hot: (30000, 52)
[]


,LIMIT_BAL,AGE,PAY_0,PAY_2,PAY_3,PAY_4,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,...,PAY_6_-2,PAY_6_-1,PAY_6_0,PAY_6_2,PAY_6_3,PAY_6_4,PAY_6_5,PAY_6_6,PAY_6_7,PAY_6_8
0,20000,24,2,2,-1,-1,3913,3102,689,0,...,True,False,False,False,False,False,False,False,False,False
1,120000,26,-1,2,0,0,2682,1725,2682,3272,...,False,False,False,True,False,False,False,False,False,False
2,90000,34,0,0,0,0,29239,14027,13559,14331,...,False,False,True,False,False,False,False,False,False,False
3,50000,37,0,0,0,0,46990,48233,49291,28314,...,False,False,True,False,False,False,False,False,False,False
4,50000,57,-1,0,-1,0,8617,5670,35835,20940,...,False,False,True,False,False,False,False,False,False,False


## 6. Build feature_info for GAN

In [13]:
feature_info = build_feature_info(df_encoded, cat_cols, target_col='Class')
column_order = df_encoded.drop(columns=['Class']).columns.tolist()

print("Numerical features:", feature_info["numerical"])
print("Categorical groups:", list(feature_info["categorical"].keys()))

Numerical features: ['LIMIT_BAL', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
Categorical groups: ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_5', 'PAY_6']


## 7. Save everything

In [15]:
df_encoded.to_csv("../data/processed/default_credit_onehot.csv", index=False)
joblib.dump(feature_info, "../models/feature_info_default_credit.pkl")
joblib.dump(column_order, "../models/column_order_default_credit.pkl")
joblib.dump(cat_cols, "../models/cat_cols_default_credit.pkl")
print("All saved.")

All saved.
